In [ ]:
import numpy as np
import pandas as pd
import time
import math
import os
import h5py
import matplotlib.pyplot as plt

def load_hdf5(dataset_path):
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        exit()

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

In [ ]:
# Load into df
data_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task2\episode_0.hdf5'
qpos, qvel, action, image_dict = load_hdf5(dataset_path=data_file)

# Load qpos and action into DataFrames with joint headers
joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']

# Create timestamp column (assuming sequential timesteps)
num_timesteps = len(qpos)
timestamps = np.arange(num_timesteps)

# Create qpos DataFrame
qpos_data = np.column_stack([timestamps, qpos])
qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)

# Create action DataFrame
action_data = np.column_stack([timestamps, action])
action_df = pd.DataFrame(action_data, columns=joint_headers)

print("QPos DataFrame shape:", qpos_df.shape)
print("Action DataFrame shape:", action_df.shape)
print("\nQPos DataFrame head:")
print(qpos_df.head())
print("\nAction DataFrame head:")
print(action_df.head())

# Optional: Check the data ranges for each joint
print("\nQPos data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {qpos_df[col].min():.3f} to {qpos_df[col].max():.3f}")

print("\nAction data ranges:")
for col in joint_headers[1:]:  # Skip timestamp
    print(f"{col}: {action_df[col].min():.3f} to {action_df[col].max():.3f}")

In [ ]:
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt

def fix_temporal_alignment_with_first_qpos(qpos_df, action_df, joint_headers):
    """
    Fix temporal alignment by:
    1. Copy first row of QPos and insert it at the beginning (duplicate first position)
    2. Copy QPos[t+1] to Action[t] (actions should predict next states) 
    3. Truncate to maintain same length
    
    Result: QPos becomes [qpos[0], qpos[0], qpos[1], qpos[2], ...]
            Action becomes [qpos[0], qpos[1], qpos[2], qpos[3], ...]
    """
    
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Create corrected dataframes
    qpos_corrected = qpos_df.copy()
    action_corrected = action_df.copy()
    
    # Step 1: Copy first row of QPos and insert at beginning
    first_qpos_row = qpos_df.iloc[0].copy()
    first_qpos_row['timestamp'] = 0  # Set timestamp to 0 for the duplicated row
    
    # Insert the first row at the beginning
    qpos_corrected = pd.concat([pd.DataFrame([first_qpos_row]), qpos_corrected], ignore_index=True)
    
    # Step 2: Copy QPos[t+1] to Action[t] (actions predict next states)
    # Now qpos_corrected has: [qpos[0], qpos[0], qpos[1], qpos[2], ...]
    # We want action to be:     [qpos[0], qpos[1], qpos[2], qpos[3], ...]
    for joint in joint_cols:
        # Use the shifted QPos as action targets
        action_corrected[joint] = qpos_corrected[joint][1:len(action_corrected)+1].values
    
    # Step 3: Truncate QPos to maintain same length as action
    qpos_corrected = qpos_corrected[:-1].reset_index(drop=True)
    
    # Update timestamps to be sequential
    qpos_corrected['timestamp'] = range(len(qpos_corrected))
    action_corrected['timestamp'] = range(len(action_corrected))
    
    return qpos_corrected, action_corrected

# Apply the fix
print("Applying temporal alignment fix using first QPos row...")
qpos_fixed, action_fixed = fix_temporal_alignment_with_first_qpos(qpos_df, action_df, joint_headers)

print(f"Original QPos shape: {qpos_df.shape}")
print(f"Original Action shape: {action_df.shape}")
print(f"Fixed QPos shape: {qpos_fixed.shape}")
print(f"Fixed Action shape: {action_fixed.shape}")

print("\nInitial positions check:")
print("Original QPos first row:")
print(qpos_df.iloc[0])
print("\nFixed QPos first row (should be copy of original first row):")
print(qpos_fixed.iloc[0])
print("\nFixed QPos second row (should be same as first row):")
print(qpos_fixed.iloc[1])
print("\nFixed Action first row (should match original QPos first row):")
print(action_fixed.iloc[0])
print("\nFixed Action second row (should match original QPos second row):")
print(action_fixed.iloc[1])

# Verify the mapping
print("\nVerification of QPos->Action mapping:")
print("Format: QPos[t] -> Action[t] (should predict QPos[t+1])")
print("-" * 60)
joint_cols = ['b', 's', 'e', 't', 'r', 'g']
for i in range(min(5, len(qpos_fixed)-1)):
    print(f"Timestep {i}:")
    for joint in joint_cols[:3]:  # Show first 3 joints for brevity
        qpos_t = qpos_fixed[joint].iloc[i]
        action_t = action_fixed[joint].iloc[i]
        qpos_t_plus_1 = qpos_fixed[joint].iloc[i+1]
        match = "✓" if abs(action_t - qpos_t_plus_1) < 1e-6 else "✗"
        print(f"  {joint}: QPos[{i}]={qpos_t:.3f} -> Action[{i}]={action_t:.3f} (should equal QPos[{i+1}]={qpos_t_plus_1:.3f}) {match}")
    print()

In [ ]:
# Visualize the relationship for all joints to verify the fix
joint_cols = ['b', 's', 'e', 't', 'r', 'g']
joint_names = ['Base', 'Shoulder', 'Elbow', 'Wrist', 'Roll', 'Gripper']

# Create comparison plots: Original vs Fixed
fig, axes = plt.subplots(3, 4, figsize=(20, 15))

for i, (joint, joint_name) in enumerate(zip(joint_cols, joint_names)):
    row = i // 2
    col_orig = (i % 2) * 2
    col_fixed = col_orig + 1
    
    # Original data (first 100 points for clarity)
    n_points = min(100, len(qpos_df))
    axes[row, col_orig].plot(qpos_df['timestamp'][:n_points], qpos_df[joint][:n_points], 
                            'b-', linewidth=2, label='Original QPos', alpha=0.8)
    axes[row, col_orig].plot(action_df['timestamp'][:n_points], action_df[joint][:n_points], 
                            'r-', linewidth=2, label='Original Action', alpha=0.8)
    axes[row, col_orig].set_title(f'{joint_name} - Original (WRONG)')
    axes[row, col_orig].set_xlabel('Timestamp')
    axes[row, col_orig].set_ylabel(f'{joint} Value')
    axes[row, col_orig].legend()
    axes[row, col_orig].grid(True, alpha=0.3)
    
    # Fixed data
    axes[row, col_fixed].plot(qpos_fixed['timestamp'][:n_points], qpos_fixed[joint][:n_points], 
                             'b-', linewidth=2, label='Fixed QPos', alpha=0.8)
    axes[row, col_fixed].plot(action_fixed['timestamp'][:n_points], action_fixed[joint][:n_points], 
                             'r-', linewidth=2, label='Fixed Action', alpha=0.8)
    axes[row, col_fixed].set_title(f'{joint_name} - Fixed (CORRECT)')
    axes[row, col_fixed].set_xlabel('Timestamp')
    axes[row, col_fixed].set_ylabel(f'{joint} Value')
    axes[row, col_fixed].legend()
    axes[row, col_fixed].grid(True, alpha=0.3)

plt.tight_layout()
plt.show()

In [ ]:
def save_corrected_hdf5(qpos_fixed, action_fixed, qvel, image_dict, output_path, cfg=None):
    """
    Save the corrected qpos, action data along with unchanged qvel and image_dict 
    back to HDF5 format matching the original structure with optimized settings
    """
    
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Convert DataFrames back to numpy arrays (excluding timestamp column)
    qpos_corrected_array = qpos_fixed[joint_cols].values
    action_corrected_array = action_fixed[joint_cols].values
    
    # Ensure qvel matches the length of corrected data
    qvel_corrected = qvel[:len(qpos_corrected_array)]
    
    # Ensure image data matches the length of corrected data
    image_dict_corrected = {}
    for cam_name, images in image_dict.items():
        image_dict_corrected[cam_name] = images[:len(qpos_corrected_array)]
    
    max_timesteps = len(qpos_corrected_array)
    
    print(f"Saving corrected data:")
    print(f"  QPos shape: {qpos_corrected_array.shape}")
    print(f"  Action shape: {action_corrected_array.shape}")
    print(f"  QVel shape: {qvel_corrected.shape}")
    for cam_name, images in image_dict_corrected.items():
        print(f"  {cam_name} images shape: {images.shape}")
    
    # Create the HDF5 file with optimized settings matching the recording script
    with h5py.File(output_path, 'w', rdcc_nbytes=1024 ** 2 * 2) as root:
        # Set simulation attribute
        root.attrs['sim'] = True
        
        # Create observations group
        obs = root.create_group('observations')
        
        # Create images group with optimized chunking
        image = obs.create_group('images')
        for cam_name, images in image_dict_corrected.items():
            cam_height, cam_width = images.shape[1], images.shape[2]
            _ = image.create_dataset(cam_name, 
                                   (max_timesteps, cam_height, cam_width, 3), 
                                   dtype='uint8',
                                   chunks=(1, cam_height, cam_width, 3))
        
        # Create qpos and qvel datasets with proper dimensions
        state_dim = qpos_corrected_array.shape[1]  # Should be 6
        action_dim = action_corrected_array.shape[1]  # Should be 6
        
        qpos = obs.create_dataset('qpos', (max_timesteps, state_dim))
        qvel = obs.create_dataset('qvel', (max_timesteps, state_dim))
        action = root.create_dataset('action', (max_timesteps, action_dim))
        
        # Create data dictionary to match the original format
        data_dict = {
            '/observations/qpos': qpos_corrected_array,
            '/observations/qvel': qvel_corrected,
            '/action': action_corrected_array,
        }
        
        # Add image data to data_dict
        for cam_name, images in image_dict_corrected.items():
            data_dict[f'/observations/images/{cam_name}'] = images
        
        # Write all data using the same pattern as recording script
        for name, array in data_dict.items():
            root[name][...] = array
    
    print(f"✓ Corrected data saved to: {output_path}")
    
    # Verify the saved file
    print("\nVerification - loading saved file:")
    qpos_verify, qvel_verify, action_verify, image_dict_verify = load_hdf5(output_path)
    print(f"  Loaded QPos shape: {qpos_verify.shape}")
    print(f"  Loaded Action shape: {action_verify.shape}")
    print(f"  Loaded QVel shape: {qvel_verify.shape}")
    for cam_name, images in image_dict_verify.items():
        print(f"  Loaded {cam_name} images shape: {images.shape}")

# Save the corrected data
output_file = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task2c\episode_0_c.hdf5'
save_corrected_hdf5(qpos_fixed, action_fixed, qvel, image_dict, output_file)

In [ ]:
import os
import glob
import numpy as np
import pandas as pd
import h5py
from pathlib import Path

def process_all_episodes():
    """
    Process all HDF5 files in task2 folder and save corrected versions to task2c folder
    """
    
    # Define paths
    input_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task2'
    output_dir = r'C:\Users\Administrator\Documents\transformer\ACT-Shaka\data\task2c'
    
    # Create output directory if it doesn't exist
    os.makedirs(output_dir, exist_ok=True)
    
    # Find all HDF5 files in the input directory
    hdf5_files = glob.glob(os.path.join(input_dir, '*.hdf5'))
    
    if not hdf5_files:
        print(f"No HDF5 files found in {input_dir}")
        return
    
    print(f"Found {len(hdf5_files)} HDF5 files to process:")
    for file in hdf5_files:
        print(f"  {os.path.basename(file)}")
    
    # Process each file
    for i, input_file in enumerate(hdf5_files):
        print(f"\n{'='*60}")
        print(f"Processing file {i+1}/{len(hdf5_files)}: {os.path.basename(input_file)}")
        print(f"{'='*60}")
        
        try:
            # Load the original data
            print("Loading original data...")
            qpos, qvel, action, image_dict = load_hdf5(input_file)
            
            # Convert to DataFrames
            joint_headers = ['timestamp', 'b', 's', 'e', 't', 'r', 'g']
            
            # Create timestamp column
            num_timesteps = len(qpos)
            timestamps = np.arange(num_timesteps)
            
            # Create DataFrames
            qpos_data = np.column_stack([timestamps, qpos])
            qpos_df = pd.DataFrame(qpos_data, columns=joint_headers)
            
            action_data = np.column_stack([timestamps, action])
            action_df = pd.DataFrame(action_data, columns=joint_headers)
            
            print(f"  Original QPos shape: {qpos_df.shape}")
            print(f"  Original Action shape: {action_df.shape}")
            print(f"  Original QVel shape: {qvel.shape}")
            for cam_name, images in image_dict.items():
                print(f"  Original {cam_name} images shape: {images.shape}")
            
            # Apply the temporal alignment fix
            print("Applying temporal alignment fix...")
            qpos_fixed, action_fixed = fix_temporal_alignment_with_first_qpos(qpos_df, action_df, joint_headers)
            
            # Generate output filename
            input_filename = os.path.basename(input_file)
            output_filename = input_filename
            output_file = os.path.join(output_dir, output_filename)
            
            # Save the corrected data
            print(f"Saving corrected data to: {output_filename}")
            save_corrected_hdf5(qpos_fixed, action_fixed, qvel, image_dict, output_file)
            
            print(f"✅ Successfully processed {input_filename}")
            
        except Exception as e:
            print(f"❌ Error processing {os.path.basename(input_file)}: {str(e)}")
            continue
    
    print(f"\n{'='*60}")
    print("PROCESSING COMPLETE")
    print(f"{'='*60}")
    print(f"Input directory: {input_dir}")
    print(f"Output directory: {output_dir}")
    
    # List the output files
    output_files = glob.glob(os.path.join(output_dir, '*.hdf5'))
    print(f"Generated {len(output_files)} corrected files:")
    for file in output_files:
        print(f"  {os.path.basename(file)}")

def fix_temporal_alignment_with_first_qpos(qpos_df, action_df, joint_headers):
    """
    Fix temporal alignment by:
    1. Copy first row of QPos and insert it at the beginning (duplicate first position)
    2. Copy QPos[t+1] to Action[t] (actions should predict next states) 
    3. Truncate to maintain same length
    
    Result: QPos becomes [qpos[0], qpos[0], qpos[1], qpos[2], ...]
            Action becomes [qpos[0], qpos[1], qpos[2], qpos[3], ...]
    """
    
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Create corrected dataframes
    qpos_corrected = qpos_df.copy()
    action_corrected = action_df.copy()
    
    # Step 1: Copy first row of QPos and insert at beginning
    first_qpos_row = qpos_df.iloc[0].copy()
    first_qpos_row['timestamp'] = 0  # Set timestamp to 0 for the duplicated row
    
    # Insert the first row at the beginning
    qpos_corrected = pd.concat([pd.DataFrame([first_qpos_row]), qpos_corrected], ignore_index=True)
    
    # Step 2: Copy QPos[t+1] to Action[t] (actions predict next states)
    for joint in joint_cols:
        # Use the shifted QPos as action targets
        action_corrected[joint] = qpos_corrected[joint][1:len(action_corrected)+1].values
    
    # Step 3: Truncate QPos to maintain same length as action
    qpos_corrected = qpos_corrected[:-1].reset_index(drop=True)
    
    # Update timestamps to be sequential
    qpos_corrected['timestamp'] = range(len(qpos_corrected))
    action_corrected['timestamp'] = range(len(action_corrected))
    
    return qpos_corrected, action_corrected

def load_hdf5(dataset_path):
    """Load HDF5 file and return qpos, qvel, action, and image_dict"""
    if not os.path.isfile(dataset_path):
        print(f'Dataset does not exist at \n{dataset_path}\n')
        raise FileNotFoundError(f"File not found: {dataset_path}")

    with h5py.File(dataset_path, 'r') as root:
        qpos = root['/observations/qpos'][()]
        qvel = root['/observations/qvel'][()]
        image_dict = dict()
        for cam_name in root[f'/observations/images/'].keys():
            image_dict[cam_name] = root[f'/observations/images/{cam_name}'][()]
        action = root['/action'][()]

    return qpos, qvel, action, image_dict

def save_corrected_hdf5(qpos_fixed, action_fixed, qvel, image_dict, output_path):
    """
    Save the corrected qpos, action data along with unchanged qvel and image_dict 
    back to HDF5 format matching the original structure with optimized settings
    """
    
    joint_cols = ['b', 's', 'e', 't', 'r', 'g']
    
    # Convert DataFrames back to numpy arrays (excluding timestamp column)
    qpos_corrected_array = qpos_fixed[joint_cols].values
    action_corrected_array = action_fixed[joint_cols].values
    
    # Ensure qvel matches the length of corrected data
    qvel_corrected = qvel[:len(qpos_corrected_array)]
    
    # Ensure image data matches the length of corrected data
    image_dict_corrected = {}
    for cam_name, images in image_dict.items():
        image_dict_corrected[cam_name] = images[:len(qpos_corrected_array)]
    
    max_timesteps = len(qpos_corrected_array)
    
    # Create the HDF5 file with optimized settings matching the recording script
    with h5py.File(output_path, 'w', rdcc_nbytes=1024 ** 2 * 2) as root:
        # Set simulation attribute
        root.attrs['sim'] = True
        
        # Create observations group
        obs = root.create_group('observations')
        
        # Create images group with optimized chunking
        image = obs.create_group('images')
        for cam_name, images in image_dict_corrected.items():
            cam_height, cam_width = images.shape[1], images.shape[2]
            _ = image.create_dataset(cam_name, 
                                   (max_timesteps, cam_height, cam_width, 3), 
                                   dtype='uint8',
                                   chunks=(1, cam_height, cam_width, 3))
        
        # Create qpos and qvel datasets with proper dimensions
        state_dim = qpos_corrected_array.shape[1]  # Should be 6
        action_dim = action_corrected_array.shape[1]  # Should be 6
        
        qpos = obs.create_dataset('qpos', (max_timesteps, state_dim))
        qvel = obs.create_dataset('qvel', (max_timesteps, state_dim))
        action = root.create_dataset('action', (max_timesteps, action_dim))
        
        # Create data dictionary to match the original format
        data_dict = {
            '/observations/qpos': qpos_corrected_array,
            '/observations/qvel': qvel_corrected,
            '/action': action_corrected_array,
        }
        
        # Add image data to data_dict
        for cam_name, images in image_dict_corrected.items():
            data_dict[f'/observations/images/{cam_name}'] = images
        
        # Write all data using the same pattern as recording script
        for name, array in data_dict.items():
            root[name][...] = array

# Run the batch processing
if __name__ == "__main__":
    process_all_episodes()